# 摘要:

伴随着基于物理的渲染在实时渲染领域广泛使用，区域光在渲染中的重要性也变得越来越强。对一些特定类型的区域光源以及BRDF，存在一个数学上的解析解，然后这些解析解并不能使用在Microfacet BRDF上，目前存在的一些求解方向包括Most Representative Point, Linearly Transform Cosine (LTC), Analytic spherical harmonic coefficients for polygonal area lights等。其中LTC由于其精确性，较好的性能，多种光源类型的支持成为游戏等实时渲染应用程序的首选方案。
然而要将LTC在移动平台的生产项目中使用仍然存在不少挑战。例如horizon clipping 问题，三角形与平面求交从而得到新的多边形，然后在shader里实现这样的函数会造成大量的代码分支以及寄存器的占用。还有区域光的裁切问题，如何能够找到一个区域光的几何代理用来与模型进行求交计算？如何在tiled/clustered renderer上对区域光做binning？另外还有移动平台上浮点数计算精度的问题，在移动平台上大量的计算将在FP16上进行，然而从之前的实践中我们得知LTC对计算精度有着很高的要求。还有区域光如果做Level of detail管理？我们将对这一系列在实践中将会问题的展开讨论。


# Polygonal integration over Lambert surface

多边形在半球面的cosine分布函数下
$$
{\frac 1 \pi} {\int_{P} cos(\overrightarrow{\rm n},\overrightarrow{\rm \omega}) } d\omega
$$

存在解析解为:
$$
E(p_1, ..., p_n) = \frac 1 {2 \pi} \sum_{i=1}^{n} acos(\langle p_i, p_j \rangle) \langle {\frac {p_i \times p_j} {\| {p_i \times p_j} \|} }, {\begin{bmatrix} 0\\ 0\\ 1\\ \end{bmatrix}} \rangle \quad \textrm{where} \quad j = i + 1
$$

原论文中仅给出公式，这里对这个公式给出具体推导。推导主要是使用了[斯托克斯定理](https://en.wikipedia.org/wiki/Stokes%27_theorem)。
$$
\iint _{S}\nabla \times \mathbf {F} \,\cdot \,d\mathbf {S} = \oint _{\Gamma }\mathbf {F} \,\cdot \,d{\mathbf {\Gamma } }
$$

首先我们希望将等式的的左边转换为公式1等价的形式，因为他们的积分域相同，都是球面。斯托克斯定理描述的是一个向量场的旋度与表面法线的点乘。
因为是在球面上，所以任意点的表面法线即为原点到当前点的向量，即为公式1中的$\omega$. 由于两个向量的点乘就是这两个向量的cosine函数，所以如果我们将一个旋度为(0,0,1)的向量场函数代入到斯托克斯定理左边的式子，我们将得到公式1。
对于一个向量场函数$F=(F_x, F_y, F_z)$，对应的旋度定义如下：
$$
TODO
$$

我们希望得到一个在半球面上旋度恒为(0,0,1)的向量场函数，一个简单的例子为$F(x,y,z)=(0,x,0)$

斯托克斯等式的右边为一个边界积分项，其中对多边形的积分项等于对各个边界的积分项求和。我们将这个积分项通过Slerp参数化将得到
$$
{\displaystyle \operatorname {Slerp} (p_{0},p_{1};t)={\frac {\sin {[(1-t)\Omega }]}{\sin \Omega }}p_{0}+{\frac {\sin[t\Omega ]}{\sin \Omega }}p_{1}.}
$$


# LUT fitting and normalization

最终生成的LTC矩阵存在5个变量，下面是对这5个变量得直观展示以及其几何意义:  
m00 = 0.5, 缩放x轴
![](images/ltc_m00_0.5.png)

m02 = 0.33, 在ZX平面上旋转
![](images/ltc_m02_0.33.png)

m11 = 0.5, 缩放y轴
![](images/ltc_m11_0.5.png)

m20 = 0.5, 偏移度
![](images/ltc_m20_0.85.png)

m22 = 3, 缩放z轴
![](images/ltc_m22_3.png)

In [ ]:
import numpy as np
import ipyvolume as ipv
from matplotlib import cm, colors
import ipywidgets as widgets

fig = ipv.figure(lighting=False)


def spherical_dir(theta, phi):
    x = np.sin(theta) * np.cos(phi)
    y = np.sin(theta) * np.sin(phi)
    z = np.cos(theta)
    return x, y, z

def spherical_coord(x, y, z):
    norm = np.sqrt(x**2 + y**2 + z**2)
    theta = np.arccos(z/norm)
    phi = np.arctan2(y, x)
    return theta, phi

def meshgrid_spherical_coord(numsamples):
    theta = np.linspace(0, np.pi, num=numsamples)
    phi = np.linspace(0, 2*np.pi, num=numsamples*2)
    theta, phi = np.meshgrid(theta, phi)
    return theta, phi

def D(Wi):
    return np.maximum(Wi[2], 0) / np.pi


def D_ltc(Wi, transfo):
    x = Wi[0]
    y = Wi[1]
    z = Wi[2]
    
    inv_transfo = np.linalg.inv(transfo)
    
    x_orig = x * inv_transfo[0][0] + y * inv_transfo[0][1] + z * inv_transfo[0][2]
    y_orig = x * inv_transfo[1][0] + y * inv_transfo[1][1] + z * inv_transfo[1][2]
    z_orig = x * inv_transfo[2][0] + y * inv_transfo[2][1] + z * inv_transfo[2][2]

    l = np.sqrt(x_orig**2 + y_orig**2 + z_orig**2)

    # vector normalization
    x_orig /= l
    y_orig /= l
    z_orig /= l

    # evaluate spherical function
    vals = D(np.array([x_orig, y_orig, z_orig]))
    
    # apply change of variable
    jacobian = np.linalg.det(inv_transfo) / (l*l*l);
    vals *= jacobian
    
    return vals

def plot_ltc_func(transfo):
    theta, phi = meshgrid_spherical_coord(64)
    x, y, z = spherical_dir(theta, phi)
    
    vals = D_ltc(np.array([x, y, z]), transfo)

    # normalize the value for better visualization
    vals /= np.max(vals)
    ipv.plot_mesh(x, z, y, wireframe=False, color=cm.coolwarm(vals))
#     ipv.plot_mesh(x*vals, z*vals, y*vals, wireframe=False, color=cm.coolwarm(vals))

    # plot lines in order to visualize the geometry transform
    sample_points = np.array([[0, 0, 2], [-1, -1, 2], [-1, 1, 2], [1, 1, 2], [1, -1, 2]])
    for p in sample_points:
        p = np.dot(transfo, p)
        p /= np.linalg.norm(p)
        p *= 2
        ipv.plot([0, p[0]], [0, p[2]], [0, p[1]])


def plot(a, b, c, d, e):
    transfo = np.identity(3, dtype = float)        
    transfo[0][0] = a
    transfo[0][2] = b
    transfo[1][1] = c
    transfo[2][0] = d
    transfo[2][2] = e

    fig.meshes.clear()
    fig.scatters.clear()
    plot_ltc_func(transfo)
    ipv.xyzlim(-2, 2)


aSlider = widgets.FloatSlider(min=0.01, max=1, step=0.01, value=1)
bSlider = widgets.FloatSlider(min=-1, max=1, step=0.01, value=0)
cSlider = widgets.FloatSlider(min=0.01, max=1, step=0.01, value=1)
dSlider = widgets.FloatSlider(min=-1, max=1, step=0.01, value=0)
eSlider = widgets.FloatSlider(min=0.01, max=4, step=0.01, value=1)
widgets.interact(plot, a=aSlider, b=bSlider, c=cSlider, d=dSlider, e=eSlider)
ipv.show()

然而要直接求解这个5维的优化问题还是很困难得，我们要想办法降低问题的复杂性，首先我们先尝试将GGX替换成Phong模型，在Phong模型的情况下，可以将cosine lobe通过缩放与旋转矩阵来达到对phong的拟合，其中旋转矩阵的Z轴为反射向量，由于Phong模型是各项同性的，所以缩放矩阵的m00与m11相等，问题简化为单变量的优化问题，就很容易解决了。下图为两个矩阵变换得直观显示。

In [ ]:
def plot_phong(theta, scale):
    scale_mat = np.identity(3, dtype = float)        
    scale_mat[0][0] = scale
    scale_mat[1][1] = scale
    rotate_mat = np.identity(3, dtype = float)
    rotate_mat[2] = np.array([-np.sin(theta), 0, np.cos(theta)])
    rotate_mat[0] = np.cross(rotate_mat[1], rotate_mat[2])
    transfo = rotate_mat @ scale_mat

    fig.meshes.clear()
    fig.scatters.clear()
    plot_ltc_func(transfo)
    ipv.xyzlim(-2, 2)


thetaSlider = widgets.FloatSlider(min=0, max=np.pi*0.5, step=0.01, value=0)
scaleSlider = widgets.FloatSlider(min=0.01, max=1, step=0.01, value=1)
widgets.interact(plot_phong, theta=thetaSlider, scale=scaleSlider)
ipv.show()

有了上面的经验之后，我们再尝试将Phong模型替换成GGX。下图为Phong与GGX lobe的对比:

从上面我们可以观察到，GGX为各项异性且lobe的主要方向与反射向量存在一定夹角。我们继续按照将矩阵分解成旋转与缩放矩阵的思路进行求解，其中旋转矩阵与Phong模型一致。此时通过我们观察到的GGX对比Phong的性质，可以通过m00与m11来对X,Y方向的缩放以及m20对YZ平面上的旋转来达到GGX的近似，具体效果见下图：

最终我们通过将LTC变换矩阵分解成旋转矩阵以及本地变换矩阵（X,Y方向的缩放以及YZ平面上的旋转），将一个5维的优化问题简化成3维的优化问题。

TODO: LUT fitting implementation

最终生成的LUT需要存储5个变量，这样需要两次贴图查找操作，我们希望将变量控制在4个以内，这样只需要一次查表操作。让我们回顾一下LTC中关于D的解析式
$$
D(\omega) = D_o({\omega}_o) {\frac {\partial {\omega}_o} {\partial \omega}} 
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {| M^{-1} |} {{\| {|M^{-1} \omega} \|}^3}}
$$

其中$M^{-1}$为我们生成的LUT表。通过上的解析式分析我们可以得到，将$\lambda I M^{-1}$替换掉$M^{-1}$，上面表达式依然成立, 其中$I$为单位矩阵。
$$
D(\omega) = D_o (\frac {\lambda I  M^{-1} \omega} {\| \lambda I {M^{-1} \omega} \|}) {\frac {| \lambda I M^{-1} |} {{\| \lambda I {M^{-1} \omega} \|}^3}}  \\
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {{| \lambda I|} {| M^{-1} |}} { {{\lambda}^3} {{\| {M^{-1} \omega} \|}^3}} } \\
          = D_o (\frac {M^{-1} \omega} {\| {M^{-1} \omega} \|}) {\frac {{{\lambda}^3} {| M^{-1} |}} { {{\lambda}^3} {{\| {M^{-1} \omega} \|}^3}} }
$$

所以我们可以将LUT矩阵除以m00, m11, m22任意一个来达到将数据压缩成4个的目的。下图为分别使用m00, m11, m22来进行归一化后的数据展示。

# Naïve implementation


# Horizon clipping

第一章节中提到的多边形在lambert表面的积分是对clamped cosine函数进行求积，所以我们需要将多边形裁切到上半球，shader里实现多边形的裁切会造成大量的代码分支，以下为伪代码示例。

```
    /* Detect clipping config */
    int config = 0;
    if(L[0].z > 0.0) config += 1;
    if(L[1].z > 0.0) config += 2;
    if(L[2].z > 0.0) config += 4;
    if(L[3].z > 0.0) config += 8;

    if(config == 0) {
        // clip all
    } else if(config == 1) { // V1 clip V2 V3 V4
        ...
    } else if(config == 2) { // V2 clip V1 V3 V4
        ...
    ...
    } else if(config == 14) { // V2 V3 V4 clip V1
        ...
    } else if(config == 15) { // V1 V2 V3 V4
        n = 4;
    }
```

原论文并没有对裁切这部分的实践做介绍，不过Stephen Hill在[Real-Time Area Lighting:a Journey from Research to Production](https://advances.realtimerendering.com/s2016/s2016_ltc_rnd.pdf)中给出了一个近似方案，这里做一下简单的介绍。

我们可以将horizon clipping的求解分解成下面两个部分
polygonal_integration_without_horizon_clipping * horizon_clipping_factor
其中polygonal_integration_without_horizon_clipping可以直接求解公式1得到，我们的目标就变成求解horizon_clipping_factor。这时候的思路是通过更简单的几何代理来近似这个值，最简单的几何代理就是球了，球与平面的horizon clipping存在解析解。于是这时候问题转变成如何选择球的半径与朝向。

如果将公式1中与法线向量(<0,0,1>向量)的点乘去掉，那么我们会得到这个form factor的向量形式，这个向量形式与另外一个单位向量U的点乘结果为当前的form factor在以U为Z轴的cosine球面分布函数上的积分。我们可以将vector form factor的方向做为球的朝向，同时选择form factor相同的球。通过下面的公式求解得到, 详细推导参考[sphere ambient occlusion](https://www.iquilezles.org/www/articles/sphereao/sphereao.htm)
$$
angular\_extent = asin(sqrt(length(F)))
$$

这样我们就得到了这个球的张角$sigma$和球与法线的夹角$omega$，这种情况horizon clipping factor存在解析解如下
$$
TODO
$$

由于计算还是过于复杂，我们可以将这个结果保存为贴图，在运行时求$sigma$与$omega$，再通过这两个参数通过查表的方式来求解。

TODO: LUT polynomial fit

由于使用球来近似平面，无法处理平面的朝向问题，所以我们需要自己来处理平面的朝向，将当前位置带入平面方程就可以得到当前位置处于 平面的正面或者方面，通过对F向量取反即可处理双面光源。

# Geometry proxy (TODO)